In [1]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('playground-series-s6e9/train.csv')
test = pd.read_csv('playground-series-s6e9/test.csv')
sample_sub = pd.read_csv('playground-series-s6e9/sample_submission.csv')

# Preprocessing
target = 'Will_Buy_EV'
features = [c for c in train.columns if c not in ['id', target]]

cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train[target] = train[target].map({'Yes': 1, 'No': 0})

X = train[features]
y = train[target]
X_test = test[features]

num_trials = 50
n_splits = 5


In [2]:
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': 'balanced',
        'n_jobs': -1,
        'random_state': 42
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = RandomForestClassifier(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, preds)
        auc_scores.append(auc)
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=num_trials)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params['class_weight'] = 'balanced'
best_params['n_jobs'] = -1
best_params['random_state'] = 42

with open('random_forest_best_params.json', 'w') as f:
    json.dump(best_params, f)


[I 2026-09-11 11:35:53,700] A new study created in memory with name: no-name-0a0985dd-8846-419d-88e3-dad41a84fd61


[I 2026-09-11 11:39:54,278] Trial 0 finished with value: 0.9330524024433038 and parameters: {'n_estimators': 151, 'max_depth': 5, 'min_samples_split': 17, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 0 with value: 0.9330524024433038.


[I 2026-09-11 11:46:53,988] Trial 1 finished with value: 0.9386445813145601 and parameters: {'n_estimators': 440, 'max_depth': 9, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.9386445813145601.


[I 2026-09-11 11:55:00,288] Trial 2 finished with value: 0.9397315563493068 and parameters: {'n_estimators': 467, 'max_depth': 13, 'min_samples_split': 16, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 2 with value: 0.9397315563493068.


[I 2026-09-11 12:03:14,357] Trial 3 finished with value: 0.9404516031073363 and parameters: {'n_estimators': 215, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 19, 'max_features': None}. Best is trial 3 with value: 0.9404516031073363.


...

[I 2026-09-11 21:59:59,055] Trial 46 finished with value: 0.9398248829862691 and parameters: {'n_estimators': 457, 'max_depth': 16, 'min_samples_split': 17, 'min_samples_leaf': 19, 'max_features': 'log2'}. Best is trial 43 with value: 0.9405850699823542.


[I 2026-09-11 22:20:54,434] Trial 47 finished with value: 0.940513378462412 and parameters: {'n_estimators': 487, 'max_depth': 16, 'min_samples_split': 20, 'min_samples_leaf': 16, 'max_features': None}. Best is trial 43 with value: 0.9405850699823542.


[I 2026-09-11 22:42:39,680] Trial 48 finished with value: 0.9405051709252756 and parameters: {'n_estimators': 468, 'max_depth': 13, 'min_samples_split': 20, 'min_samples_leaf': 14, 'max_features': None}. Best is trial 43 with value: 0.9405850699823542.


[I 2026-09-11 23:06:53,755] Trial 49 finished with value: 0.9404410763279948 and parameters: {'n_estimators': 486, 'max_depth': 12, 'min_samples_split': 17, 'min_samples_leaf': 15, 'max_features': None}. Best is trial 43 with value: 0.9405850699823542.


Best trial: {'n_estimators': 499, 'max_depth': 15, 'min_samples_split': 19, 'min_samples_leaf': 20, 'max_features': None}


In [3]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_preds_df = pd.DataFrame({'id': test['id']})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = RandomForestClassifier(**best_params)
    model.fit(X_tr, y_tr)
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    fold_test_preds = model.predict_proba(X_test)[:, 1]
    test_preds += fold_test_preds / n_splits
    fold_preds_df[f'fold_{fold}'] = fold_test_preds

print("OOF AUC:", roc_auc_score(y, oof_preds))

pd.DataFrame({'id': train['id'], 'Will_Buy_EV': oof_preds}).to_csv('random_forest_oof_predictions.csv', index=False)
fold_preds_df.to_csv('random_forest_fold_predictions.csv', index=False)

sample_sub['Will_Buy_EV'] = test_preds
sample_sub.to_csv('submission_random_forest.csv', index=False)


OOF AUC: 0.9405711607508763
